# VEPL Multi-GPU DDP Run On Kaggle

Use this notebook with Kaggle's 2-GPU accelerator, for example T4 x2 when available. The goal is to validate the cloud multi-GPU `DistributedDataParallel` path on the real VEPL UAV dataset.

## Kaggle Setup

In the notebook right sidebar:

- Accelerator: `GPU T4 x2` or another 2-GPU option.
- Internet: `On`, because the notebook clones GitHub and downloads VEPL from Zenodo.

Then run cells top-to-bottom.

In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
print('gpu count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

## Clone Repository

In [ ]:
%cd /kaggle/working
!rm -rf multi-gpu-vegetation-risk-vision-pipeline
!git clone https://github.com/msa-1988/multi-gpu-vegetation-risk-vision-pipeline.git
%cd /kaggle/working/multi-gpu-vegetation-risk-vision-pipeline

## Install Lightweight Dependencies

Kaggle normally has CUDA-enabled PyTorch preinstalled. Installing only lightweight packages avoids replacing the working torch build.

In [ ]:
!pip install -q numpy tqdm matplotlib

import os
os.environ['PYTHONPATH'] = '/kaggle/working/multi-gpu-vegetation-risk-vision-pipeline/src'

## Download VEPL

In [ ]:
!bash scripts/download_vepl_sample.sh
!PYTHONPATH=src python scripts/visualize_vepl_samples.py --samples 4 --image-size 256 --output artifacts/figures/kaggle_vepl_samples.png

In [ ]:
from IPython.display import Image, display
display(Image('artifacts/figures/kaggle_vepl_samples.png'))

## Run 2-GPU DDP Training

This launches two processes with `torchrun`. The script writes a single rank-0 artifact under `artifacts/runs/kaggle_2gpu`.

In [ ]:
!PYTHONPATH=src bash scripts/run_kaggle_2gpu.sh

## Inspect Results

In [ ]:
!PYTHONPATH=src python scripts/compare_runs.py
!PYTHONPATH=src python scripts/plot_training_curves.py --metrics artifacts/runs/kaggle_2gpu/metrics.json --output artifacts/figures/kaggle_training_curves.png
!PYTHONPATH=src python scripts/visualize_vepl_predictions.py \
  --checkpoint artifacts/runs/kaggle_2gpu/best_model.pt \
  --output artifacts/figures/kaggle_vepl_predictions.png \
  --image-size 192 \
  --base-channels 32 \
  --samples 4 \
  --device cuda

display(Image('artifacts/figures/kaggle_training_curves.png'))
display(Image('artifacts/figures/kaggle_vepl_predictions.png'))

## What To Report

Open `artifacts/runs/kaggle_2gpu/metrics.json` and report:

- `world_size`: should be `2`.
- `device`: should be `cuda:0` from rank 0.
- best validation IoU/F1 from the history.
- images/sec compared with local single-GPU training.